In [1]:
import os

folders = [
    "/home/jovyan/retail-data-warehouse/data/bronze",
    "/home/jovyan/retail-data-warehouse/data/silver",
    "/home/jovyan/retail-data-warehouse/data/gold",
    "/home/jovyan/retail-data-warehouse/ingestion",
    "/home/jovyan/retail-data-warehouse/transformation",
    "/home/jovyan/retail-data-warehouse/warehouse",
    "/home/jovyan/retail-data-warehouse/analytics",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ Created: {folder}")

print("\n🎉 Project structure ready!")

✅ Created: /home/jovyan/retail-data-warehouse/data/bronze
✅ Created: /home/jovyan/retail-data-warehouse/data/silver
✅ Created: /home/jovyan/retail-data-warehouse/data/gold
✅ Created: /home/jovyan/retail-data-warehouse/ingestion
✅ Created: /home/jovyan/retail-data-warehouse/transformation
✅ Created: /home/jovyan/retail-data-warehouse/warehouse
✅ Created: /home/jovyan/retail-data-warehouse/analytics

🎉 Project structure ready!


In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

# ── Generate Customers ──
customers = pd.DataFrame({
    'customer_id': range(1, 101),
    'customer_name': [f'Customer_{i}' for i in range(1, 101)],
    'city': np.random.choice(['Mumbai', 'Delhi', 'Pune', 'Bangalore', 'Chennai'], 100),
    'segment': np.random.choice(['Retail', 'Wholesale', 'Online'], 100),
    'join_date': [
        (datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1000))).strftime('%Y-%m-%d')
        for _ in range(100)
    ]
})

# ── Generate Products ──
products = pd.DataFrame({
    'product_id': range(1, 51),
    'product_name': [f'Product_{i}' for i in range(1, 51)],
    'category': np.random.choice(['Electronics', 'Clothing', 'Food', 'Furniture', 'Sports'], 50),
    'brand': np.random.choice(['BrandA', 'BrandB', 'BrandC', 'BrandD'], 50),
    'unit_price': np.round(np.random.uniform(10, 500, 50), 2)
})

# ── Generate Stores ──
stores = pd.DataFrame({
    'store_id': range(1, 11),
    'store_name': [f'Store_{i}' for i in range(1, 11)],
    'city': np.random.choice(['Mumbai', 'Delhi', 'Pune', 'Bangalore', 'Chennai'], 10),
    'store_type': np.random.choice(['Mall', 'Standalone', 'Online'], 10),
    'opening_date': [
        (datetime(2015, 1, 1) + timedelta(days=random.randint(0, 2000))).strftime('%Y-%m-%d')
        for _ in range(10)
    ]
})

# ── Generate Sales ──
sales = pd.DataFrame({
    'sale_id': range(1, 1001),
    'customer_id': np.random.randint(1, 101, 1000),
    'product_id': np.random.randint(1, 51, 1000),
    'store_id': np.random.randint(1, 11, 1000),
    'quantity': np.random.randint(1, 20, 1000),
    'discount_pct': np.random.choice([0, 5, 10, 15, 20], 1000),
    'sale_date': [
        (datetime(2023, 1, 1) + timedelta(days=random.randint(0, 730))).strftime('%Y-%m-%d')
        for _ in range(1000)
    ]
})

# ── Save to Bronze Layer ──
BASE = "/home/jovyan/retail-data-warehouse/data/bronze"
customers.to_csv(f"{BASE}/customers.csv", index=False)
products.to_csv(f"{BASE}/products.csv", index=False)
stores.to_csv(f"{BASE}/stores.csv", index=False)
sales.to_csv(f"{BASE}/sales.csv", index=False)

print("✅ Bronze layer loaded!")
print(f"   Customers : {len(customers)} rows")
print(f"   Products  : {len(products)} rows")
print(f"   Stores    : {len(stores)} rows")
print(f"   Sales     : {len(sales)} rows")

✅ Bronze layer loaded!
   Customers : 100 rows
   Products  : 50 rows
   Stores    : 10 rows
   Sales     : 1000 rows


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, upper, trim, when

spark = SparkSession.builder \
    .appName("RetailDWH") \
    .getOrCreate()

BASE_BRONZE = "/home/jovyan/retail-data-warehouse/data/bronze"
BASE_SILVER = "/home/jovyan/retail-data-warehouse/data/silver"

# ── Clean Customers ──
customers = spark.read.csv(f"{BASE_BRONZE}/customers.csv", header=True, inferSchema=True)
customers_clean = customers \
    .withColumn("customer_name", trim(col("customer_name"))) \
    .withColumn("city", upper(col("city"))) \
    .withColumn("segment", trim(col("segment"))) \
    .withColumn("join_date", to_date(col("join_date"), "yyyy-MM-dd")) \
    .dropna()

# ── Clean Products ──
products = spark.read.csv(f"{BASE_BRONZE}/products.csv", header=True, inferSchema=True)
products_clean = products \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("category", upper(col("category"))) \
    .withColumn("brand", trim(col("brand"))) \
    .withColumn("unit_price", col("unit_price").cast("double")) \
    .dropna()

# ── Clean Stores ──
stores = spark.read.csv(f"{BASE_BRONZE}/stores.csv", header=True, inferSchema=True)
stores_clean = stores \
    .withColumn("store_name", trim(col("store_name"))) \
    .withColumn("city", upper(col("city"))) \
    .withColumn("store_type", trim(col("store_type"))) \
    .withColumn("opening_date", to_date(col("opening_date"), "yyyy-MM-dd")) \
    .dropna()

# ── Clean Sales ──
sales = spark.read.csv(f"{BASE_BRONZE}/sales.csv", header=True, inferSchema=True)
sales_clean = sales \
    .withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd")) \
    .withColumn("quantity", col("quantity").cast("integer")) \
    .withColumn("discount_pct", col("discount_pct").cast("double")) \
    .filter(col("quantity") > 0) \
    .dropna()

# ── Save to Silver Layer ──
customers_clean.write.csv(f"{BASE_SILVER}/customers", header=True, mode="overwrite")
products_clean.write.csv(f"{BASE_SILVER}/products", header=True, mode="overwrite")
stores_clean.write.csv(f"{BASE_SILVER}/stores", header=True, mode="overwrite")
sales_clean.write.csv(f"{BASE_SILVER}/sales", header=True, mode="overwrite")

print("✅ Silver layer ready!")
print(f"   Customers : {customers_clean.count()} rows")
print(f"   Products  : {products_clean.count()} rows")
print(f"   Stores    : {stores_clean.count()} rows")
print(f"   Sales     : {sales_clean.count()} rows")

spark.stop()

✅ Silver layer ready!
   Customers : 100 rows
   Products  : 50 rows
   Stores    : 10 rows
   Sales     : 1000 rows


In [4]:
import pandas as pd
import glob
import os

BASE_SILVER = "/home/jovyan/retail-data-warehouse/data/silver"
BASE_GOLD = "/home/jovyan/retail-data-warehouse/data/gold"

def read_silver(table):
    files = glob.glob(f"{BASE_SILVER}/{table}/part-*.csv")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

# ── Read Silver Data ──
customers = read_silver("customers")
products = read_silver("products")
stores = read_silver("stores")
sales = read_silver("sales")

# ── DimCustomer ──
dim_customer = customers[['customer_id', 'customer_name', 'city', 'segment', 'join_date']].copy()
dim_customer['customer_key'] = range(1, len(dim_customer) + 1)

# ── DimProduct ──
dim_product = products[['product_id', 'product_name', 'category', 'brand', 'unit_price']].copy()
dim_product['product_key'] = range(1, len(dim_product) + 1)

# ── DimStore ──
dim_store = stores[['store_id', 'store_name', 'city', 'store_type', 'opening_date']].copy()
dim_store['store_key'] = range(1, len(dim_store) + 1)

# ── DimDate ──
all_dates = pd.to_datetime(sales['sale_date']).dt.date.unique()
dim_date = pd.DataFrame({'full_date': sorted(all_dates)})
dim_date['date_key'] = range(1, len(dim_date) + 1)
dim_date['day'] = pd.to_datetime(dim_date['full_date']).dt.day
dim_date['month'] = pd.to_datetime(dim_date['full_date']).dt.month
dim_date['quarter'] = pd.to_datetime(dim_date['full_date']).dt.quarter
dim_date['year'] = pd.to_datetime(dim_date['full_date']).dt.year
dim_date['month_name'] = pd.to_datetime(dim_date['full_date']).dt.strftime('%B')
dim_date['day_name'] = pd.to_datetime(dim_date['full_date']).dt.strftime('%A')
dim_date['is_weekend'] = pd.to_datetime(dim_date['full_date']).dt.dayofweek >= 5

# ── FactSales ──
fact_sales = sales.copy()
fact_sales['sale_date'] = pd.to_datetime(fact_sales['sale_date']).dt.date

# Join keys
fact_sales = fact_sales.merge(dim_customer[['customer_id', 'customer_key']], on='customer_id')
fact_sales = fact_sales.merge(dim_product[['product_id', 'product_key', 'unit_price']], on='product_id')
fact_sales = fact_sales.merge(dim_store[['store_id', 'store_key']], on='store_id')
fact_sales = fact_sales.merge(dim_date[['full_date', 'date_key']], left_on='sale_date', right_on='full_date')

# Calculate measures
fact_sales['gross_amount'] = fact_sales['quantity'] * fact_sales['unit_price']
fact_sales['discount_amount'] = fact_sales['gross_amount'] * (fact_sales['discount_pct'] / 100)
fact_sales['net_amount'] = fact_sales['gross_amount'] - fact_sales['discount_amount']

# Keep only necessary columns
fact_sales = fact_sales[[
    'sale_id', 'customer_key', 'product_key', 'store_key', 'date_key',
    'quantity', 'discount_pct', 'gross_amount', 'discount_amount', 'net_amount'
]]

# ── Save Gold Layer ──
os.makedirs(BASE_GOLD, exist_ok=True)
dim_customer.to_csv(f"{BASE_GOLD}/dim_customer.csv", index=False)
dim_product.to_csv(f"{BASE_GOLD}/dim_product.csv", index=False)
dim_store.to_csv(f"{BASE_GOLD}/dim_store.csv", index=False)
dim_date.to_csv(f"{BASE_GOLD}/dim_date.csv", index=False)
fact_sales.to_csv(f"{BASE_GOLD}/fact_sales.csv", index=False)

print("✅ Gold Layer — Star Schema Ready!")
print(f"   DimCustomer : {len(dim_customer)} rows")
print(f"   DimProduct  : {len(dim_product)} rows")
print(f"   DimStore    : {len(dim_store)} rows")
print(f"   DimDate     : {len(dim_date)} rows")
print(f"   FactSales   : {len(fact_sales)} rows")

✅ Gold Layer — Star Schema Ready!
   DimCustomer : 100 rows
   DimProduct  : 50 rows
   DimStore    : 10 rows
   DimDate     : 560 rows
   FactSales   : 1000 rows


In [5]:
print("=== DimCustomer ===")
print(dim_customer.head(3).to_string())

print("\n=== DimProduct ===")
print(dim_product.head(3).to_string())

print("\n=== DimStore ===")
print(dim_store.head(3).to_string())

print("\n=== DimDate ===")
print(dim_date.head(3).to_string())

print("\n=== FactSales ===")
print(fact_sales.head(3).to_string())

=== DimCustomer ===
   customer_id customer_name       city    segment   join_date  customer_key
0            1    Customer_1  BANGALORE     Online  2021-10-16             1
1            2    Customer_2    CHENNAI     Retail  2020-04-24             2
2            3    Customer_3       PUNE  Wholesale  2020-01-26             3

=== DimProduct ===
   product_id product_name     category   brand  unit_price  product_key
0           1    Product_1  ELECTRONICS  BrandD      497.28            1
1           2    Product_2     CLOTHING  BrandD       96.20            2
2           3    Product_3     CLOTHING  BrandA       18.86            3

=== DimStore ===
   store_id store_name   city  store_type opening_date  store_key
0         1    Store_1   PUNE      Online   2016-12-28          1
1         2    Store_2  DELHI  Standalone   2016-03-05          2
2         3    Store_3  DELHI      Online   2018-10-04          3

=== DimDate ===
    full_date  date_key  day  month  quarter  year month_name

In [6]:
readme = """# Retail Sales Data Warehouse

## Overview
An end-to-end Data Warehouse project built using Medallion Architecture 
(Bronze → Silver → Gold) mirroring Azure Synapse Analytics + Azure Data Lake Storage.

## Architecture
## Azure Equivalent Stack
| Local Tool | Azure Equivalent |
|------------|-----------------|
| Python + Pandas | Azure Data Factory |
| Local folders | Azure Data Lake Storage |
| PySpark | Azure Databricks |
| SQL Server | Azure Synapse Analytics |
| Star Schema | Azure Synapse Dedicated Pool |

## Medallion Architecture
- **Bronze Layer** — Raw CSV data ingested from source systems (Customers, Products, Stores, Sales)
- **Silver Layer** — PySpark cleaned and transformed data (nulls removed, types cast, strings trimmed)
- **Gold Layer** — Star Schema dimensional model ready for analytics

## Star Schema Design
   DimDate
          │
DimProduct ───┼─── FactSales ─── DimCustomer

│

DimStore

### Tables
| Table | Type | Rows | Description |
|-------|------|------|-------------|
| DimCustomer | Dimension | 100 | Customer details with surrogate keys |
| DimProduct | Dimension | 50 | Product catalogue with categories |
| DimStore | Dimension | 10 | Store locations and types |
| DimDate | Dimension | 560 | Date dimension with day/month/quarter/year |
| FactSales | Fact | 1000 | Sales transactions with measures |

## Key Business Insights

### Revenue by Category
| Category | Revenue | Units Sold |
|----------|---------|------------|
| Food | ₹6,88,645 | 2,628 |
| Electronics | ₹5,49,411 | 2,263 |
| Clothing | ₹5,02,616 | 2,080 |
| Furniture | ₹4,30,642 | 2,176 |
| Sports | ₹2,40,831 | 915 |

### Top Customer
Customer_51 from Bangalore (Wholesale) — ₹48,416 total spend

### Best Store
Store_5 in Mumbai (Mall) — ₹2,69,475 total revenue

### Weekend vs Weekday
| Day Type | Sales | Revenue |
|----------|-------|---------|
| Weekday | 722 | ₹17,28,184 |
| Weekend | 278 | ₹6,83,963 |

## How to Run
1. Run `warehouse_pipeline.ipynb` to generate Bronze → Silver → Gold layers
2. Import Gold layer CSVs into SQL Server
3. Run analytics queries in `analytics/queries.sql`

## Key Concepts Demonstrated
- Medallion Architecture (Bronze/Silver/Gold)
- Star Schema dimensional modelling
- Surrogate key generation
- PySpark data transformation
- SQL analytical queries
- ETL pipeline design
"""

with open("/home/jovyan/retail-data-warehouse/README.md", "w") as f:
    f.write(readme)

print("✅ README created!")

✅ README created!
